# 05.2 — Comparación final: HistBoost vs CatBoost vs LSTM

Comparación reproducible usando los artefactos ya generados. El foco de la entrega final es explicar por qué el modelo tabular fuerte (`HistGradientBoosting`) supera al experimento secuencial (`LSTM`) y cómo `CatBoost` funciona como comparador alternativo muy competitivo.

Modelos incluidos si existen artefactos:
- `market_value_baseline`
- `price_sequence_gru`
- `price_sequence_lstm`
- `hist_gradient_boosting`
- `catboost_residual`


In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

MODELS_ROOT = ROOT / 'data' / 'models'
REGISTRY = MODELS_ROOT / 'registry'
MODEL_LABELS = {
    'market_value_baseline': 'MLP residual',
    'price_sequence_gru': 'GRU residual',
    'price_sequence_lstm': 'LSTM residual',
    'hist_gradient_boosting': 'HistBoost',
    'catboost_residual': 'CatBoost residual',
}
PALETTE = {
    'MLP residual': '#8D99AE',
    'GRU residual': '#457B9D',
    'LSTM residual': '#2A9D8F',
    'HistBoost': '#264653',
    'CatBoost residual': '#E76F51',
    'Mercado': '#ADB5BD',
}


## 1. Cargar métricas offline

Todos los modelos se comparan contra el mismo benchmark: `price_yes` del mercado.


In [ ]:
rows = []
metrics_by_model = {}
for model_dir in MODEL_LABELS:
    metrics_path = MODELS_ROOT / model_dir / 'test_metrics.json'
    if not metrics_path.exists():
        continue
    payload = json.loads(metrics_path.read_text())
    if not payload.get('test'):
        continue
    test = payload['test']
    label = MODEL_LABELS[model_dir]
    metrics_by_model[label] = test
    rows.append({
        'model': label,
        'beats_market_brier': payload.get('market_baseline_beaten', {}).get('brier'),
        'beats_market_log_loss': payload.get('market_baseline_beaten', {}).get('log_loss'),
        'beats_market_topk': payload.get('market_baseline_beaten', {}).get('top_k_realized_pnl'),
        'brier': test['calibrated_metrics']['brier'],
        'log_loss': test['calibrated_metrics']['log_loss'],
        'ece': test['calibrated_metrics']['ece'],
        'roc_auc': test['calibrated_metrics']['roc_auc'],
        'pr_auc': test['calibrated_metrics']['pr_auc'],
        'top_k_avg_realized_pnl': test['ev_metrics']['top_k_avg_realized_pnl'],
        'top_k_hit_rate': test['ev_metrics']['top_k_hit_rate'],
        'top_k_avg_predicted_ev': test['ev_metrics']['top_k_avg_predicted_ev'],
    })
summary_df = pd.DataFrame(rows).sort_values(['top_k_avg_realized_pnl', 'brier'], ascending=[False, True])
display(summary_df)

market = next(iter(metrics_by_model.values()))['market_baseline_metrics']
print('Mercado baseline:')
display(pd.Series({k: market[k] for k in ['brier', 'log_loss', 'ece', 'roc_auc', 'pr_auc']}))


## 2. Scorecard principal

Lectura esperada con los artefactos actuales: CatBoost y HistBoost superan al mercado; LSTM no.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
ordered = summary_df['model'].tolist()
colors = [PALETTE.get(m, '#555') for m in ordered]

axes[0].bar(ordered, summary_df.set_index('model').loc[ordered, 'brier'], color=colors)
axes[0].axhline(market['brier'], color=PALETTE['Mercado'], ls='--', label='Mercado')
axes[0].set_title('Brier ↓')
axes[0].tick_params(axis='x', rotation=25)
axes[0].legend()

axes[1].bar(ordered, summary_df.set_index('model').loc[ordered, 'log_loss'], color=colors)
axes[1].axhline(market['log_loss'], color=PALETTE['Mercado'], ls='--', label='Mercado')
axes[1].set_title('Log-loss ↓')
axes[1].tick_params(axis='x', rotation=25)

axes[2].bar(ordered, summary_df.set_index('model').loc[ordered, 'top_k_avg_realized_pnl'], color=colors)
axes[2].axhline(0, color='black', lw=1)
axes[2].set_title('Top-K avg realized PnL ↑')
axes[2].tick_params(axis='x', rotation=25)
plt.tight_layout()


## 3. Ventajas y desventajas por modelo


In [ ]:
pros_cons = pd.DataFrame([
    {
        'modelo': 'HistBoost',
        'ventajas': 'Rápido, simple, calibración estable, buenas señales live, supera al mercado en Brier/log-loss/Top-K.',
        'desventajas': 'Menos flexible con secuencias crudas; depende de features tabulares diseñadas.',
        'uso_en_presentacion': 'Modelo principal contra LSTM.'
    },
    {
        'modelo': 'CatBoost residual',
        'ventajas': 'Mejor métrica offline actual; maneja categoría directo y residual económico; Top-K muy fuerte en test.',
        'desventajas': 'En live quedó conservador: 0 señales accionables con thresholds actuales; calibración isotónica puede producir outputs discretos/extremos.',
        'uso_en_presentacion': 'Comparador alternativo fuerte / fallback.'
    },
    {
        'modelo': 'LSTM residual',
        'ventajas': 'Prueba explícita de dinámica temporal; usa trayectoria de precios y no solo snapshot.',
        'desventajas': 'No supera al mercado; Top-K PnL negativo; sobreestima EV; más complejo sin beneficio empírico.',
        'uso_en_presentacion': 'Experimento negativo que fortalece la conclusión metodológica.'
    },
])
display(pros_cons)


## 4. Desempeño por horizonte


In [ ]:
bucket_rows = []
for model, payload in metrics_by_model.items():
    for bucket, b in payload.get('by_horizon', {}).items():
        bucket_rows.append({
            'model': model,
            'bucket': bucket,
            'count': b.get('count', 0),
            'brier': b.get('probability_metrics', {}).get('brier'),
            'market_brier': b.get('market_baseline_metrics', {}).get('brier'),
            'top_k_avg_realized_pnl': b.get('ev_metrics', {}).get('top_k_avg_realized_pnl'),
            'top_k_hit_rate': b.get('ev_metrics', {}).get('top_k_hit_rate'),
        })
bucket_df = pd.DataFrame(bucket_rows)
display(bucket_df)

focus = bucket_df[bucket_df['model'].isin(['HistBoost', 'CatBoost residual', 'LSTM residual'])]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
focus.pivot(index='bucket', columns='model', values='brier').plot.bar(ax=axes[0], color=[PALETTE.get(c, '#555') for c in focus['model'].unique()])
axes[0].set_title('Brier por horizonte ↓')
axes[0].tick_params(axis='x', rotation=20)

focus.pivot(index='bucket', columns='model', values='top_k_avg_realized_pnl').plot.bar(ax=axes[1], color=[PALETTE.get(c, '#555') for c in focus['model'].unique()])
axes[1].axhline(0, color='black', lw=1)
axes[1].set_title('Top-K realized PnL por horizonte ↑')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()


## 5. Comparación live


In [ ]:
live_path = REGISTRY / 'live_scores.csv'
if live_path.exists():
    live_df = pd.read_csv(live_path)
    live_rows = []
    for raw_name, label in MODEL_LABELS.items():
        sub = live_df[live_df['model_name'] == raw_name]
        if sub.empty:
            continue
        live_rows.append({
            'model': label,
            'rows': len(sub),
            'strong_buy': int((sub['signal'] == 'STRONG BUY').sum()),
            'buy': int((sub['signal'] == 'BUY').sum()),
            'hold': int((sub['signal'] == 'HOLD').sum()),
            'median_ev': float(sub['ev_per_share'].median()),
            'p90_ev': float(sub['ev_per_share'].quantile(0.90)),
        })
    live_summary = pd.DataFrame(live_rows)
    display(live_summary)

    fig, ax = plt.subplots(figsize=(10, 4))
    live_summary.set_index('model')[['strong_buy', 'buy', 'hold']].plot.bar(stacked=True, ax=ax)
    ax.set_title('Señales live por modelo')
    ax.tick_params(axis='x', rotation=25)
    plt.tight_layout()
else:
    print('Falta live_scores.csv. Corre: python -m src.scoring.scorer --config config/config.yaml --all-models')


## 6. Hallazgos para la presentación

- **HistBoost gana claramente al LSTM**: menor Brier/log-loss y utilidad Top-K positiva.
- **CatBoost es el modelo offline más fuerte** en esta corrida, pero su política live quedó conservadora con los thresholds actuales.
- **LSTM no justifica más tuning**: early stopping temprano, no supera al mercado y su Top-K PnL es negativo.
- **La conclusión técnica es defendible**: en este dataset, las features tabulares snapshot-compatible capturan mejor el edge que una arquitectura secuencial sobre precios.
- **Narrativa recomendada**: presentar HistBoost vs LSTM como comparación central y CatBoost como benchmark alternativo que confirma que modelos tabulares/residuales son más adecuados.
